# MAIN ORCHESTRATOR
+ Retrieves and spread variables values across al the job
+ Creates the whole architecture needed to the proper functionallity of the Medallion
+ Decide which layer will be executed

In [0]:
create_architecture = dbutils.widgets.get("create_architecture")
create_orchestrator_tables = dbutils.widgets.get("create_orchestrator_tables")

API_KEY = dbutils.widgets.get("API_KEY")
SUBGRAPH_ID = dbutils.widgets.get("SUBGRAPH_ID")

In [0]:
execute_layers = []
config_table_path = "uniswap.orchestrator.configuration"
logs_table_path = "uniswap.orchestrator.logs"

In [0]:
SUBGRAPH_URL = f"https://gateway.thegraph.com/api/{API_KEY}/subgraphs/id/{SUBGRAPH_ID}"

In [0]:
if create_architecture == "true":
    print("*INFO*: Creating Medallion Architecture...")
    dbutils.notebook.run("../00_architecture/create_basic_architecture",
                         0)

if create_orchestrator_tables == "true":
    print("*INFO*: Creating orchestrator tables...")
    dbutils.notebook.run("../00_architecture/create_orchestrator_tables",
                        0,
                        {
                            "config_table_path": config_table_path,
                            "logs_table_path": logs_table_path
                        })
    
    print("*INFO*: Populating configuration table...")
    dbutils.notebook.run("./populate_config_table",
                        0,
                        {
                            "config_table_path": config_table_path,
                        })

In [0]:
config_table_df = spark.read.table(config_table_path)

In [0]:
for row in config_table_df.filter("active_for_load = true").groupBy("layer").count().collect():
    execute_layers.append(row.layer)

print(f"*INFO*: Layers to execute: {execute_layers}.")

In [0]:
dbutils.jobs.taskValues.set(key = "execute_layers", value = execute_layers)
dbutils.jobs.taskValues.set(key = "config_table_path", value = config_table_path)
dbutils.jobs.taskValues.set(key = "logs_table_path", value = logs_table_path)
dbutils.jobs.taskValues.set(key = "SUBGRAPH_URL", value = SUBGRAPH_URL)